In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict

In [ ]:
load_dotenv()  # Load environment variables from .env file
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
class BlogPostState(TypedDict):
    title: str
    outline: str
    content: str

In [ ]:
def create_outline(state: BlogPostState) -> BlogPostState:
    # fetch the title from the state
    title = state['title']

    # call the LLM to generate an outline based on the title
    prompt = f"Create a detailed outline for a blog post titled '{title}'."
    outline = model.invoke(prompt).content

    # update the state with the generated outline
    state['outline'] = outline

    return state

In [ ]:
def create_blog(state: BlogPostState) -> BlogPostState:
    # fetch the outline from the state
    title = state['title']
    outline = state['outline']

    # call the LLM to generate a blog post based on the outline
    prompt = f"Write a detailed blog post based on the title '{title}' and the following outline:\n{outline}"
    content = model.invoke(prompt).content

    # update the state with the generated content
    state['content'] = content

    return state

In [ ]:
graph = StateGraph(BlogPostState)

# nodes
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

# edges
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

workflow = graph.compile()


In [ ]:
initial_state = {'title': 'The Future of Artificial Intelligence', 'outline': '', 'content': ''}
final_state = workflow.invoke(initial_state)
print(final_state)  # Output: {'title': 'The Future of Artificial Intelligence', 'outline': '...', 'content': '...'}

In [ ]:
print(final_state['outline'])  # Output: The generated blog post content

In [ ]:
print(final_state['content'])  # Output: The generated blog post content